In [ ]:
# =============================================
# CODICE PAGERANK UTILE PER FARE IL CONFRONTO
# =============================================

import numpy as np
import scipy
from scipy import sparse
from scipy.sparse import linalg
import sklearn.utils.extmath
import pandas as pd

# =====================================================================
# FUNZIONE 1: COSTRUZIONE AUTOMATICA DI 'H' e 'a' (OTTIMIZZATA)
# =====================================================================
def prepara_strutture_sparse(sorgenti, destinazioni, n_nodi):
    """
    Costruisce la matrice sparsa H e il vettore 'a' a partire da una semplice 
    lista di link (sorgenti e destinazioni), senza mai creare matrici dense.
    """
    # 1. Creiamo un array di "1" che rappresenta l'esistenza dei link 
    dati_link = np.ones(len(sorgenti))
    
    # 2. Creiamo la Matrice di Adiacenza (A) in formato sparso (CSC - Compressed Sparse Column)
    A = sparse.csc_matrix((dati_link, (destinazioni, sorgenti)), shape=(n_nodi, n_nodi))
    
    # 3. Calcoliamo quanti link escono da ogni pagina (sommando i valori lungo le colonne)
    link_uscenti = A.sum(axis=0).A1 
    
    # 4. Creiamo il vettore 'a' dei dangling nodes in automatico!
    a = (link_uscenti == 0).astype(int)
    
    # 5. Prepariamoci a dividere per calcolare le probabilità (creiamo H)
    link_uscenti_sicuri = np.where(link_uscenti == 0, 1.0, link_uscenti)
    
    # Dividiamo le colonne della matrice sparsa A per il numero di link uscenti.
    diagonale_inversi = sparse.diags(1.0 / link_uscenti_sicuri)
    H = sklearn.utils.extmath.safe_sparse_dot(A, diagonale_inversi)
    
    return H, a 

# ================================================================================
# FUNZIONE 2: L'ALGORITMO PAGERANK MATRIX-FREE
# ===============================================================================
def pagerank_matrix_free(H, a, alpha=0.85, tol=1e-8, max_iter=100):
    """ 
    Calcola il PageRank tramite iterazione (Metodo delle potenze)
    usando un approccio Matrix-Free per gestire reti di grandi dimensioni.
    """
    n = H.shape[0] 
    
    # 1. Inizializzazione
    e = np.ones(n) 

    # Metodo delle potenze : Iterazione fino alla convergenza o al raggiungimento del numero massimo di iterazioni
    for k in range(1, max_iter + 1):
        
        # 2. Prodotto scalare v^T * a (gestione della probabilità "persa" nei dangling nodes)
        dangling_sum = v.dot(a)

        # 3. Prodotto matrice-vettore H * v 
        hyperlink_sum = sklearn.utils.extmath.safe_sparse_dot(H, v)

        # 4. Aggiornamento Matrix-Free
        v_new = alpha * hyperlink_sum + (alpha * dangling_sum + (1 - alpha)) * e / n
        
        # 5. Criterio di arresto: controlliamo se la differenza scende sotto la tolleranza
        if np.linalg.norm(v_new - v, 1) < tol:
            print(f"Convergenza raggiunta con successo all'iterazione {k}.")
            return v_new
            
        # Se la convergenza non è ancora raggiunta, aggiorniamo v per la prossima iterazione
        v = v_new
        
    print("Attenzione: convergenza non raggiunta entro il limite massimo di iterazioni.")
    return v


In [ ]:
# ===================================================================================
# CODICE CON I MINIMI QUADRATI per 100 nodi (non ottimizzato per 1 milione di nodi )
# ===================================================================================

import numpy as np
from scipy.optimize import minimize
from scipy import sparse

# 2. Definiamo la funzione obiettivo: l'errore quadratico
# Rappresenta || (I - alpha * H)x - b ||^2
def objective(x, H, alpha, b):
    n = H.shape[0]
    
    # Creazione della Matrice Identità SPARSA per evitare il LinAlgError
    I = sparse.eye(n) 
    
    # Calcolo dell'errore fedele all'equazione (I - alpha*H)x - b
    errore = (I - alpha * H).dot(x) - b 
    
    return np.sum(errore**2) #effettuo la norma quadratica dell'errore

def pagerank_minimi_quadrati(H_sparsa, alpha=0.85):
    """
    Calcola il PageRank risolvendo il sistema lineare (I - alpha*H)x = b 
    tramite l'ottimizzazione dei minimi quadrati.
    """
    n = H_sparsa.shape[0]
    
    # 1. Il termine noto b nel sistema (I - alpha * H)x = b
    # b è il vettore di "salto casuale"
    #np.ones(n) equivale a e, quindi b = (1 - alpha) * e / n
    b_vec = np.ones(n) * (1 - alpha) / n
    
    # 3. Definiamo i vincoli (La somma delle autorità deve essere 1)
    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    
    # 4. Limiti: ogni punteggio deve essere tra 0 e 1 (Bounds)
    bnds = tuple((0, 1) for _ in range(n))
    
    # Punto di partenza uniforme
    x_iniziale = np.ones(n) / n
    
    # 5. Risoluzione tramite ottimizzazione
    risultato_opt = minimize(objective, x_iniziale, args=(H_sparsa, alpha, b_vec), 
                             method='SLSQP', constraints=cons, bounds=bnds, tol=1e-8, options={'maxiter': 70}) 
    
    return risultato_opt.x

In [ ]:
# ===================================================================================
# CODICE CON I MINIMI QUADRATI ottimizzato per 1.000.000 nodi 
# ===================================================================================

from scipy.sparse.linalg import LinearOperator, lsqr
import numpy as np

def pagerank_minimi_quadrati_ottimizzato_corretto(H_sparsa, is_dangling_array, alpha=0.85):
    
    #=========================================================================================
    #Calcola il PageRank minimizzando ||A*x - b||^2 tramite LSQR.
    # Corretto con LinearOperator per calcolare dinamicamente i Dangling Nodes (BOT)
    # ed evitare probabilità negative e perdite di stocasticità.
    #=========================================================================================
      
    # 1. Ricaviamo 'N', ovvero il numero totale di nodi nella rete, 
    N = H_sparsa.shape[0]
    
    # 2. Definiamo il termine noto 'b'. 
    b = ((1.0 - alpha) / N) * np.ones(N)
    
    # ====================================================================
    # 3. MOLTIPLICAZIONE IN AVANTI (matvec: A * x)
    # Questa funzione calcola il lato sinistro dell'equazione (I - alpha*H)*x
    # ====================================================================
    def matvec(x):
        # Passo A: Calcola la propagazione normale dei link.
        # Fa il vettore corrente 'x' - alpha * H * x
        risultato_base = x - (alpha * H_sparsa.dot(x))
        
        # Passo B: Il recupero dinamico dai vicoli ciechi.
        # Usa la variabile booleana 'is_dangling_array' per sommare SOLO i punteggi attuali dei BOT.
        # in modo da calcolare dinamicamente la probabilità "persa" nei vicoli ciechi (BOT) ad ogni iterazione.
        somma_bot = np.sum(x[is_dangling_array])
        
        # Passo C: Crea il vettore di correzione.
        # Prende la probabilità intrappolata (somma_bot), le applica il fattore alpha (85%)
        # e la divide equamente tra tutti i nodi della rete.
        correzione_bot = (alpha * somma_bot / N) * np.ones(N) 
        
        # Passo D: Restituisce il risultato corretto.
        # Sottrae la correzione al risultato base, completando l'equazione stocastica.
        return risultato_base - correzione_bot
        
    # ====================================================================
    # 4. MOLTIPLICAZIONE ALL'INDIETRO (rmatvec: A^T * x)
    # Serve all'algoritmo LSQR per calcolare il gradiente (la pendenza)
    # e minimizzare l'errore quadratico ||Ax - b||^2.
    # ====================================================================
    def rmatvec(x):
        # Passo A: Moltiplicazione per la TRASPOSTA della matrice sparsa.
        # Nota il '.T' (trasposta). È una richiesta algebrica di LSQR per il gradiente.
        risultato_base = x - (alpha * H_sparsa.T.dot(x))
        
        # Passo B: Calcola la somma totale del vettore corrente.
        somma_x = np.sum(x)
        
        # Passo C: La correzione trasposta dei vicoli ciechi.
        # Moltiplicare per 'is_dangling_array' (che è un array di 1/0, True/False) 
        # fa sì che la correzione si applichi SOLO sulle righe dei BOT.
        # Questo è l'esatto contrario di prima (matvec), dove spalmavamo su tutti.
        correzione_bot = (alpha * somma_x / N) * is_dangling_array
        
        # Passo D: Restituisce il risultato trasposto corretto.
        return risultato_base - correzione_bot

    # ====================================================================
    # 5. CREAZIONE E RISOLUZIONE DEL SISTEMA
    # ====================================================================
    # Incapsuliamo le due regole matematiche dentro un 'LinearOperator'.
    A_dinamica = LinearOperator((N, N), matvec=matvec, rmatvec=rmatvec)
    
    # Avviamo il risolutore iterativo LSQR.
    risultato = lsqr(A_dinamica, b, atol=1e-08, btol=1e-08 ,show=False, iter_lim=1000, x0=np.ones(N)/N, )
    
    # Il risolutore restituisce una tupla con varie info tecniche,
    # ma a noi interessa solo il primo elemento (indice [0]), che è il vettore finale 'x'.
    x = risultato[0]
    
    # 6. Sicurezza e Normalizzazione Finale.
    # np.abs() elimina eventuali microscopiche oscillazioni vicino allo zero tipiche di LSQR.
    x = np.abs(x) 
    
    # Calcola la somma del vettore finale.
    somma_x = np.sum(x)
    
    # Sicurezza anti-crash: evita la divisione per zero se la somma è nulla (improbabile, ma sicuro).
    if somma_x == 0:
        somma_x = 1e-15 
    
    # Normalizza il vettore in modo che la somma totale faccia esattamente 1 (100%).
    x = x / somma_x
    
    return x

In [9]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST SCENARIO 1 (100 nodi)
# =====================================================================
numero_totale_nodi = 100 
alpha_val = 0.85
nome_file_input = '../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv'
nome_file_output = 'Classifica_Completa_Scenario1_Confronto_PageRank_MinQuadrati.csv'

print("--- ANALISI SCENARIO 1 ---")

# 1. Caricamento Dataset e Generazione Strutture
print(f"Caricamento del dataset: {nome_file_input}...")
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 2. Esecuzione degli Algoritmi
print("Avvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard = pagerank_matrix_free(H_sparsa, vettore_a, alpha=alpha_val)

print("Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...")
pagerank_ottimizzato = pagerank_minimi_quadrati(H_sparsa, alpha=alpha_val)

# 3. Stampa a schermo dei risultati (Top 10)
print("\n--- TOP 10 PAGERANK MATRIX-FREE ---")
classifica_std = [(i, pagerank_standard[i]) for i in range(numero_totale_nodi)]
classifica_std.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_std[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
classifica_opt = [(i, pagerank_ottimizzato[i]) for i in range(numero_totale_nodi)]
classifica_opt.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_opt[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato), 4))

# 4. Salvataggio su CSV
percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_100/'
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)

df_risultati = pd.DataFrame({
    'Nodo': range(numero_totale_nodi),
    'PR_Standard': pagerank_standard.round(6),
    'PR_Ottimizzato': pagerank_ottimizzato.round(6),
    'Perc_Standard (%)': (pagerank_standard * 100).round(2),
    'Perc_Ottimizzato (%)': (pagerank_ottimizzato * 100).round(2)
})
df_risultati = df_risultati.sort_values(by='PR_Standard', ascending=False)

percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 
print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 1 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv...
Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 99.
Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 0: 0.1561 (15.61%)
2° Posto -> Nodo 1: 0.1561 (15.61%)
3° Posto -> Nodo 2: 0.1561 (15.61%)
4° Posto -> Nodo 3: 0.0055 (0.55%)
5° Posto -> Nodo 4: 0.0055 (0.55%)
6° Posto -> Nodo 5: 0.0055 (0.55%)
7° Posto -> Nodo 6: 0.0055 (0.55%)
8° Posto -> Nodo 7: 0.0055 (0.55%)
9° Posto -> Nodo 8: 0.0055 (0.55%)
10° Posto -> Nodo 9: 0.0055 (0.55%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 2: 0.1551 (15.51%)
2° Posto -> Nodo 1: 0.1551 (15.51%)
3° Posto -> Nodo 0: 0.1551 (15.51%)
4° Posto -> Nodo 23: 0.0055 (0.55%)
5° Posto -> Nodo 20: 0.0055 (0.55%)
6° Posto -> Nodo 21: 0.0055 (0.55%)
7° Posto -> Nodo 22: 0.0055 (0.55%)
8° Posto -> Nodo 10: 0.0055 (0.55%)


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.sparse.linalg import LinearOperator, lsqr

# =====================================================================
# TEST SCENARIO 1 (1.000.000 nodi) - CASO STUDIO 1
# =====================================================================
numero_totale_nodi = 1000000 
alpha_val = 0.85

nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv'
nome_file_output = 'Classifica_Completa_Scenario1_Confronto_PageRank_MinQuadrati_1MILIONE.csv'

print("--- ANALISI SCENARIO 1 (1 MILIONE DI NODI) ---")

print(f"Caricamento del dataset: {nome_file_input}...")
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

print("\nAvvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard = pagerank_matrix_free(H_sparsa, vettore_a, alpha=alpha_val, max_iter=1000)

print("\nAvvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...")
# --- LA MODIFICA CHIAVE È QUI ---
# Trasformiamo il vettore_a in un array booleano (True/False) per la nuova funzione LSQR
is_dangling_array = (vettore_a > 0)

# Passiamo is_dangling_array alla funzione corretta con LinearOperator
pagerank_ottimizzato = pagerank_minimi_quadrati_ottimizzato_corretto(H_sparsa, is_dangling_array, alpha=alpha_val)
# -------------------------------

print("\nGenerazione veloce delle classifiche...")
df_risultati = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi),
    'Score Standard': pagerank_standard,
    'Percentuale Standard (%)': (pagerank_standard * 100),
    'Score Ottimizzato': pagerank_ottimizzato,
    'Percentuale Ottimizzato (%)': (pagerank_ottimizzato * 100)
})

# ORDINAMENTO DOPPIO: In caso di pareggio, ordina per ID Nodo crescente (dal più piccolo al più grande)
df_std_sorted = df_risultati.sort_values(by=['Score Standard', 'Nodo'], ascending=[False, True])
df_opt_sorted = df_risultati.sort_values(by=['Score Ottimizzato', 'Nodo'], ascending=[False, True])

# Stampa a schermo uniformata
print("\n--- TOP 10 PAGERANK MATRIX-FREEs ---")
for pos, (idx, row) in enumerate(df_std_sorted.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Standard']:.4e} ({row['Percentuale Standard (%)']:.6f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
for pos, (idx, row) in enumerate(df_opt_sorted.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Ottimizzato']:.4e} ({row['Percentuale Ottimizzato (%)']:.6f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato), 4))

percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_1M/'
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondamento CSV uniformato
df_risultati['Score Standard'] = df_risultati['Score Standard'].round(8)
df_risultati['Percentuale Standard (%)'] = df_risultati['Percentuale Standard (%)'].round(6)
df_risultati['Score Ottimizzato'] = df_risultati['Score Ottimizzato'].round(8)
df_risultati['Percentuale Ottimizzato (%)'] = df_risultati['Percentuale Ottimizzato (%)'].round(6)

# Ordinamento doppio finale per il CSV
df_risultati = df_risultati.sort_values(by=['Score Standard', 'Nodo'], ascending=[False, True])

percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 

print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 1 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 118.

Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...

Generazione veloce delle classifiche...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 0: Score 1.5315e-01 (15.315344%)
2° Posto -> Nodo 1: Score 1.5315e-01 (15.315344%)
3° Posto -> Nodo 2: Score 1.5315e-01 (15.315344%)
4° Posto -> Nodo 3: Score 5.4054e-07 (0.000054%)
5° Posto -> Nodo 4: Score 5.4054e-07 (0.000054%)
6° Posto -> Nodo 5: Score 5.4054e-07 (0.000054%)
7° Posto -> Nodo 6: Score 5.4054e-07 (0.000054%)
8° Posto -> Nodo 7: Score 5.4054e-07 (0.000054%)
9° Posto -> Nodo 8: Score 5.4054e-07 (0.000054%)
10° Posto -> Nodo 9: Score 5.4054e-07 (0.000054%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 0: Score 1.5315e-01 (15.315345%)
2° Posto -> Nodo 

In [6]:
import os
import pandas as pd
import numpy as np

# =====================================================================
#  TEST SCENARIO 2 (100 nodi)
# =====================================================================
numero_totale_nodi = 100 
alpha_val = 0.85
nome_file_input = '../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv'
nome_file_output = 'Classifica_Completa_Scenario2_Confronto_PageRank_MinQuadrati.csv'

print("--- ANALISI SCENARIO 2 ---")

# 1. Caricamento Dataset e Generazione Strutture
print(f"Caricamento del dataset: {nome_file_input}...")
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 2. Esecuzione degli Algoritmi
print("Avvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard = pagerank_matrix_free(H_sparsa, vettore_a, alpha=alpha_val, max_iter=1000)

print("Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...")
pagerank_ottimizzato = pagerank_minimi_quadrati(H_sparsa, alpha=alpha_val)

# 3. Stampa a schermo dei risultati (Top 10)
print("\n--- TOP 10 PAGERANK MATRIX-FREE ---")
classifica_std = [(i, pagerank_standard[i]) for i in range(numero_totale_nodi)]
classifica_std.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_std[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
classifica_opt = [(i, pagerank_ottimizzato[i]) for i in range(numero_totale_nodi)]
classifica_opt.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_opt[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato), 4))

# 4. Salvataggio su CSV
percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_100/'
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)

df_risultati = pd.DataFrame({
    'Nodo': range(numero_totale_nodi),
    'PR_Standard': pagerank_standard.round(6),
    'PR_Ottimizzato': pagerank_ottimizzato.round(6),
    'Perc_Standard (%)': (pagerank_standard * 100).round(2),
    'Perc_Ottimizzato (%)': (pagerank_ottimizzato * 100).round(2)
})
df_risultati = df_risultati.sort_values(by='PR_Standard', ascending=False)

percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 
print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 2 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv...
Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 105.
Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 1: 0.2588 (25.88%)
2° Posto -> Nodo 2: 0.2526 (25.26%)
3° Posto -> Nodo 0: 0.2475 (24.75%)
4° Posto -> Nodo 61: 0.0045 (0.45%)
5° Posto -> Nodo 71: 0.0043 (0.43%)
6° Posto -> Nodo 91: 0.0043 (0.43%)
7° Posto -> Nodo 81: 0.0043 (0.43%)
8° Posto -> Nodo 72: 0.0040 (0.40%)
9° Posto -> Nodo 92: 0.0039 (0.39%)
10° Posto -> Nodo 62: 0.0039 (0.39%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 1: 0.2588 (25.88%)
2° Posto -> Nodo 2: 0.2526 (25.26%)
3° Posto -> Nodo 0: 0.2475 (24.75%)
4° Posto -> Nodo 61: 0.0045 (0.45%)
5° Posto -> Nodo 71: 0.0043 (0.43%)
6° Posto -> Nodo 91: 0.0043 (0.43%)
7° Posto -> Nodo 81: 0.0043 (0.43%)
8° Posto -> Nodo 72: 0.0039 

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.sparse.linalg import LinearOperator, lsqr

# =====================================================================
# TEST SCENARIO 2 (1.000.000 nodi) 
# =====================================================================
numero_totale_nodi = 1000000 
alpha_val = 0.85

nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv'
nome_file_output = 'Classifica_Completa_Scenario2_Confronto_PageRank_MinQuadrati_1MILIONE.csv'

print("--- ANALISI SCENARIO 2 (1 MILIONE DI NODI) ---")

print(f"Caricamento del dataset: {nome_file_input}...")
df_2 = pd.read_csv(nome_file_input)
H_sparsa_2, vettore_a_2 = prepara_strutture_sparse(df_2['Source'].values, df_2['Target'].values, numero_totale_nodi)

print("\nAvvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard_2 = pagerank_matrix_free(H_sparsa_2, vettore_a_2, alpha=alpha_val, max_iter=1000)

print("\nAvvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...")
# --- LA MODIFICA È QUI ---
# Trasformiamo il vettore_a in un array di True/False. 
# Se il valore è > 0, significa che è un vicolo cieco (True). Altrimenti è False.
is_dangling_array = (vettore_a_2 > 0)

# Ora passiamo is_dangling_array alla nuova funzione
pagerank_ottimizzato_2 = pagerank_minimi_quadrati_ottimizzato_corretto(H_sparsa_2, is_dangling_array, alpha=alpha_val)
# -------------------------

print("\nGenerazione veloce delle classifiche...")
df_risultati_2 = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi),
    'Score Standard': pagerank_standard_2,
    'Percentuale Standard (%)': (pagerank_standard_2 * 100),
    'Score Ottimizzato': pagerank_ottimizzato_2,
    'Percentuale Ottimizzato (%)': (pagerank_ottimizzato_2 * 100)
})

df_std_sorted_2 = df_risultati_2.sort_values(by='Score Standard', ascending=False)
df_opt_sorted_2 = df_risultati_2.sort_values(by='Score Ottimizzato', ascending=False)

# Stampa a schermo uniformata
print("\n--- TOP 10 PAGERANK MATRIX-FREE ---")
for pos, (idx, row) in enumerate(df_std_sorted_2.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Standard']:.4e} ({row['Percentuale Standard (%)']:.6f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
for pos, (idx, row) in enumerate(df_opt_sorted_2.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Ottimizzato']:.4e} ({row['Percentuale Ottimizzato (%)']:.6f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard_2), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato_2), 4))

percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_1M/'
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondamento CSV uniformato
df_risultati_2['Score Standard'] = df_risultati_2['Score Standard'].round(8)
df_risultati_2['Percentuale Standard (%)'] = df_risultati_2['Percentuale Standard (%)'].round(6)
df_risultati_2['Score Ottimizzato'] = df_risultati_2['Score Ottimizzato'].round(8)
df_risultati_2['Percentuale Ottimizzato (%)'] = df_risultati_2['Percentuale Ottimizzato (%)'].round(6)

df_risultati_2 = df_risultati_2.sort_values(by='Score Standard', ascending=False)

percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati_2.to_csv(percorso_completo_output, index=False) 

print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 2 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 18.

Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...

Generazione veloce delle classifiche...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 0: Score 2.7041e-01 (27.040638%)
2° Posto -> Nodo 1: Score 2.7041e-01 (27.040638%)
3° Posto -> Nodo 2: Score 2.7041e-01 (27.040638%)
4° Posto -> Nodo 926685: Score 6.0929e-07 (0.000061%)
5° Posto -> Nodo 803710: Score 5.9972e-07 (0.000060%)
6° Posto -> Nodo 728546: Score 5.9468e-07 (0.000059%)
7° Posto -> Nodo 814422: Score 5.8142e-07 (0.000058%)
8° Posto -> Nodo 944144: Score 5.7732e-07 (0.000058%)
9° Posto -> Nodo 509260: Score 5.7698e-07 (0.000058%)
10° Posto -> Nodo 939872: Score 5.7408e-07 (0.000057%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 0: Score 2.7041e

In [16]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST SCENARIO 3 (100 nodi)
# =====================================================================
numero_totale_nodi = 100 
alpha_val = 0.85
nome_file_input = '../DataSet_CasoStudio1/Rete_100/dataset_scenario3.csv'
nome_file_output = 'Classifica_Completa_Scenario3_Confronto_PageRank_MinQuadrati.csv'

print("--- ANALISI SCENARIO 3 ---")

# 1. Caricamento Dataset e Generazione Strutture
print(f"Caricamento del dataset: {nome_file_input}...")
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 2. Esecuzione degli Algoritmi
print("Avvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard = pagerank_matrix_free(H_sparsa, vettore_a, alpha=alpha_val)

print("Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...")
pagerank_ottimizzato = pagerank_minimi_quadrati(H_sparsa, alpha=alpha_val)

# 3. Stampa a schermo dei risultati (Top 10)
print("\n--- TOP 10 PAGERANK MATRIX-FREE ---")
classifica_std = [(i, pagerank_standard[i]) for i in range(numero_totale_nodi)]
classifica_std.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_std[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
classifica_opt = [(i, pagerank_ottimizzato[i]) for i in range(numero_totale_nodi)]
classifica_opt.sort(key=lambda x: x[1], reverse=True)
for pos, (nodo, pr) in enumerate(classifica_opt[:10]):
    print(f"{pos + 1}° Posto -> Nodo {nodo}: {pr:.4f} ({pr*100:.2f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato), 4))

# 4. Salvataggio su CSV
percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_100/'
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)

df_risultati = pd.DataFrame({
    'Nodo': range(numero_totale_nodi),
    'PR_Standard': pagerank_standard.round(6),
    'PR_Ottimizzato': pagerank_ottimizzato.round(6),
    'Perc_Standard (%)': (pagerank_standard * 100).round(2),
    'Perc_Ottimizzato (%)': (pagerank_ottimizzato * 100).round(2)
})
df_risultati = df_risultati.sort_values(by='PR_Standard', ascending=False)

percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 
print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 3 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario3.csv...
Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 42.
Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati)...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 1: 0.1031 (10.31%)
2° Posto -> Nodo 0: 0.0939 (9.39%)
3° Posto -> Nodo 2: 0.0750 (7.50%)
4° Posto -> Nodo 45: 0.0604 (6.04%)
5° Posto -> Nodo 80: 0.0562 (5.62%)
6° Posto -> Nodo 55: 0.0489 (4.89%)
7° Posto -> Nodo 25: 0.0373 (3.73%)
8° Posto -> Nodo 10: 0.0292 (2.92%)
9° Posto -> Nodo 15: 0.0282 (2.82%)
10° Posto -> Nodo 95: 0.0239 (2.39%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 2: 0.0948 (9.48%)
2° Posto -> Nodo 0: 0.0878 (8.78%)
3° Posto -> Nodo 1: 0.0819 (8.19%)
4° Posto -> Nodo 80: 0.0511 (5.11%)
5° Posto -> Nodo 45: 0.0500 (5.00%)
6° Posto -> Nodo 55: 0.0301 (3.01%)
7° Posto -> Nodo 25: 0.0300 (3.00%)
8° Posto -> Nodo 95: 0.0259 (2.59%

In [9]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST SCENARIO 3 (1.000.000 nodi) 
# =====================================================================
numero_totale_nodi = 1000000 
alpha_val = 0.85

nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv'
nome_file_output = 'Classifica_Completa_Scenario3_Confronto_PageRank_MinQuadrati_1MILIONE.csv'

print("--- ANALISI SCENARIO 3 (1 MILIONE DI NODI) ---")

print(f"Caricamento del dataset: {nome_file_input}...")
df_3 = pd.read_csv(nome_file_input)
H_sparsa_3, vettore_a_3 = prepara_strutture_sparse(df_3['Source'].values, df_3['Target'].values, numero_totale_nodi)

print("\nAvvio del calcolo del PageRank Matrix-Free (Standard)...")
pagerank_standard_3 = pagerank_matrix_free(H_sparsa_3, vettore_a_3, alpha=alpha_val, max_iter=1000)

print("\nAvvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...")
# --- LA MODIFICA CHIAVE È QUI ---
# Trasformiamo il vettore_a in un array booleano (True/False) per la nuova funzione LSQR
is_dangling_array_3 = (vettore_a_3 > 0)

# Passiamo is_dangling_array_3 alla funzione corretta con LinearOperator
pagerank_ottimizzato_3 = pagerank_minimi_quadrati_ottimizzato_corretto(H_sparsa_3, is_dangling_array_3, alpha=alpha_val)
# -------------------------------

print("\nGenerazione veloce delle classifiche...")
df_risultati_3 = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi),
    'Score Standard': pagerank_standard_3,
    'Percentuale Standard (%)': (pagerank_standard_3 * 100),
    'Score Ottimizzato': pagerank_ottimizzato_3,
    'Percentuale Ottimizzato (%)': (pagerank_ottimizzato_3 * 100)
})

df_std_sorted_3 = df_risultati_3.sort_values(by='Score Standard', ascending=False)
df_opt_sorted_3 = df_risultati_3.sort_values(by='Score Ottimizzato', ascending=False)

# Stampa a schermo uniformata
print("\n--- TOP 10 PAGERANK MATRIX-FREE ---")
for pos, (idx, row) in enumerate(df_std_sorted_3.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Standard']:.4e} ({row['Percentuale Standard (%)']:.6f}%)")

print("\n--- TOP 10 PAGERANK OTTIMIZZATO ---")
for pos, (idx, row) in enumerate(df_opt_sorted_3.head(10).iterrows()):
    print(f"{pos + 1}° Posto -> Nodo {int(row['Nodo'])}: Score {row['Score Ottimizzato']:.4e} ({row['Percentuale Ottimizzato (%)']:.6f}%)")

print("\nVerifica Normalizzazione Standard: ", round(np.sum(pagerank_standard_3), 4))
print("Verifica Normalizzazione Ottimizzato: ", round(np.sum(pagerank_ottimizzato_3), 4))

percorso_risultati = '../Risultati_CasoStudio1_MinQuad/Rete_1M/'
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondamento CSV uniformato
df_risultati_3['Score Standard'] = df_risultati_3['Score Standard'].round(8)
df_risultati_3['Percentuale Standard (%)'] = df_risultati_3['Percentuale Standard (%)'].round(6)
df_risultati_3['Score Ottimizzato'] = df_risultati_3['Score Ottimizzato'].round(8)
df_risultati_3['Percentuale Ottimizzato (%)'] = df_risultati_3['Percentuale Ottimizzato (%)'].round(6)

df_risultati_3 = df_risultati_3.sort_values(by='Score Standard', ascending=False)
percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati_3.to_csv(percorso_completo_output, index=False) 

print(f"\nFile CSV di confronto salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 3 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free (Standard)...
Convergenza raggiunta con successo all'iterazione 36.

Avvio del calcolo del PageRank Ottimizzato (Minimi Quadrati Corretto)...

Generazione veloce delle classifiche...

--- TOP 10 PAGERANK MATRIX-FREE ---
1° Posto -> Nodo 0: Score 1.2255e-01 (12.255167%)
2° Posto -> Nodo 1: Score 8.9917e-02 (8.991676%)
3° Posto -> Nodo 2: Score 6.0927e-02 (6.092722%)
4° Posto -> Nodo 100000: Score 1.5756e-03 (0.157559%)
5° Posto -> Nodo 91000: Score 1.5697e-03 (0.156965%)
6° Posto -> Nodo 770000: Score 1.5584e-03 (0.155838%)
7° Posto -> Nodo 830000: Score 1.5584e-03 (0.155837%)
8° Posto -> Nodo 636000: Score 1.5561e-03 (0.155610%)
9° Posto -> Nodo 983000: Score 1.5555e-03 (0.155554%)
10° Posto -> Nodo 664000: Score 1.2400e-03 (0.124004%)

--- TOP 10 PAGERANK OTTIMIZZATO ---
1° Posto -> Nodo 0: Score 1.2262e-01